[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/templates/09_causal_attention.ipynb)

# 🔴 Hard: Causal Self-Attention

Implement **causal (masked) self-attention** — the attention used in GPT-style decoders.

Same as softmax attention, but each position can **only attend to itself and earlier positions** (no peeking at future tokens).

$$\text{scores}_{ij} = \begin{cases} \frac{Q_i \cdot K_j}{\sqrt{d_k}} & \text{if } j \le i \\ -\infty & \text{if } j > i \end{cases}$$

### Signature
```python
def causal_attention(Q, K, V):
    # Q, K, V: (batch, seq, d) → output: (batch, seq, d_v)
```

### Rules
- Do **NOT** use `F.scaled_dot_product_attention`
- Position $i$ can only attend to positions $\le i$
- You **may** use `torch.softmax`, `torch.bmm`, `torch.triu`

In [1]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


In [2]:
import torch
import math

In [5]:
# ✏️ YOUR IMPLEMENTATION HERE

def causal_attention(Q, K, V):
   B,T,D=Q.shape
   mask=torch.ones(B,T,T,device=Q.device)
   mask=torch.triu(mask,diagonal=1)
   inf_mask=torch.where(mask==1,-float("inf"),0)
   score=Q@K.transpose(-1,-2)/math.sqrt(D)
   attention=torch.softmax(score+inf_mask,dim=-1)@V
   return attention


In [6]:
# 🧪 Debug
torch.manual_seed(0)
Q = torch.randn(1, 4, 8)
K = torch.randn(1, 4, 8)
V = torch.randn(1, 4, 8)
out = causal_attention(Q, K, V)
print("Output shape:", out.shape)          # (1, 4, 8)
print("Pos 0 == V[0]?", torch.allclose(out[:, 0], V[:, 0], atol=1e-5))  # should be True

Output shape: torch.Size([1, 4, 8])
Pos 0 == V[0]? True


In [7]:
from torch_judge import check
check('causal_attention')


🧪 Testing: Causal Self-Attention (Hard)
──────────────────────────────────────────────────
  ✅ [1/4] Output shape (17.1ms)
  ✅ [2/4] Future tokens don't affect past (10.5ms)
  ✅ [3/4] First position only sees itself (3.1ms)
  ✅ [4/4] Gradient flow (19.7ms)
──────────────────────────────────────────────────
  🎉 All 4 tests passed! (50.4ms total)
  Progress saved. Run status() to see your dashboard.

